## Early Stopping + Checkpoint — 训练到什么时候该停？

### 你有没有遇到过这些问题？

- 训练了 100 个 epoch，发现第 87 个 epoch 的结果才是最好的
- 训练过程中验证 loss 不降反升，但不知道什么时候该停
- 训练到一半被中断，之前的结果全丢了

**这就是 Early Stopping 和 Checkpoint 要解决的问题。**

### 本 Notebook 做什么？

1. **Early Stopping**：自动监控验证 loss，N 轮不降就停——不浪费时间
2. **Checkpoint 保存**：只保存验证集上最优的模型——不过拟合
3. **对照实验**：对比无 Early Stopping 的同一模型，展示差距

> 预期结果：Early Stopping 在第 ~20 epoch 自动停止，准确率与跑满 100 epoch 持平甚至更高。

## 1. 导入依赖与数据准备

导入 PyTorch 核心模块，自动检测 GPU。MNIST 数据从 OpenML 加载，独立运行无需依赖其他 notebook。

In [1]:
# [1.1 导入依赖]
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, random_split
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
import warnings
import os
import copy
warnings.filterwarnings('ignore')

# ==== 中文字体配置 ====
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'设备: {device}')

In [2]:
# [1.2 加载 MNIST 数据]
# 此 cell 确保 notebook 可独立运行
mnist = fetch_openml(
    name="mnist_784", version=1, as_frame=False,
    cache=True, data_home="../data"
)
X = mnist.data.reshape(-1, 28, 28).astype(np.uint8)
y = mnist.target.astype(np.uint8)
X_train, X_test = X[:60000], X[60000:]
y_train, y_test = y[:60000], y[60000:]

# 展平 + 归一化
X_train_t = torch.tensor(X_train.reshape(-1, 784), dtype=torch.float32) / 255.0
X_test_t  = torch.tensor(X_test.reshape(-1, 784),  dtype=torch.float32) / 255.0
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t  = torch.tensor(y_test,  dtype=torch.long)

# ======== 超参数集中修改区 ========
BATCH_SIZE = 64
# =================================

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test_t,  y_test_t),  batch_size=BATCH_SIZE, shuffle=False)

print(f'训练样本: {len(y_train_t):,} | 测试样本: {len(y_test_t):,}')
print(f'训练批次数: {len(train_loader)} | 测试批次数: {len(test_loader)}')

## 2. 定义 MLP 模型

与 `02_mlp_training.ipynb` 相同的 MLP 结构，方便对照实验。

In [3]:
# [2. MLP 模型定义]
class MLP(nn.Module):
    """多层感知机 — 784 -> 512 -> 256 -> 10"""
    def __init__(self, input_dim=784, hidden_dims=None, num_classes=10, dropout=0.2):
        super(MLP, self).__init__()
        if hidden_dims is None:
            hidden_dims = [512, 256]

        layers = []
        prev_dim = input_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, h_dim))
            layers.append(nn.BatchNorm1d(h_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_dim = h_dim
        layers.append(nn.Linear(prev_dim, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        if x.dim() == 4:
            x = x.view(x.size(0), -1)
        return self.net(x)


model = MLP(hidden_dims=[512, 256]).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'总参数量: {total_params:,}')
print(model)

> **`EarlyStopping` 类已提取至 `utils/early_stopping.py`**，可在任何 notebook 中通过> `from utils.early_stopping import EarlyStopping` 直接导入使用。下面展示完整实现帮助理解原理。

> **`EarlyStopping` 类已提取至 `utils/early_stopping.py`**，可在任何 notebook 中通过> `from utils.early_stopping import EarlyStopping` 直接导入使用。下面展示完整实现帮助理解原理。

## 3. 核心概念：Early Stopping 和 Checkpoint

### 为什么需要 Early Stopping？

看一张图就明白了：

```
Loss
 │
 │  训练 loss（持续下降）
 │  ╲
 │   ╲＿＿＿＿＿＿＿
 │              验证 loss
 │              ╱        ╲
 │  ──────────╱           ╲────────
 │          ↑                    ↑
 │      最佳点              过拟合开始
 └────────────────────────────────────── Epoch
```

- 训练 loss 永远在下降（模型在"背诵"训练集）
- 验证 loss 先降后升（过拟合的信号）
- **Early Stopping 在验证 loss 不再下降时自动停止**——在最佳点停下来

### Checkpoint 保存的是什么？

不是最后那个 epoch 的模型（可能已经过拟合），而是**验证集上表现最好的模型**：

```python
torch.save({
    'epoch': epoch,              # 第几轮
    'model_state_dict': ...,     # 模型权重
    'optimizer_state_dict': ..., # 优化器状态（可从断点恢复）
    'best_val_loss': ...,        # 最优验证 loss
}, 'best_checkpoint.pth')
```

### Early Stopping 的三个参数

| 参数 | 含义 | 推荐值 |
|------|------|--------|
| `patience` | 连续 N 个 epoch 验证 loss 不降则停 | 5~10（小数据集 5，大数据集 10） |
| `min_delta` | 多少变化才算"有提升" | 1e-4（太小等于没耐心，太大容易误停） |
| `mode` | 监控指标是"求小"还是"求大" | loss 用 min，acc 用 max |

In [4]:
# 直接使用共享工具类（推荐）:
import sys; sys.path.insert(0, '..')
from utils.early_stopping import EarlyStopping
print('已从 utils.early_stopping 导入 EarlyStopping')

# 以下是类实现的完整代码，仅为学习目的展示:
# 实际使用时只需上面两行 import 即可

# 直接使用共享工具类（推荐）:
import sys; sys.path.insert(0, '..')
from utils.early_stopping import EarlyStopping
print('已从 utils.early_stopping 导入 EarlyStopping')

# 以下是类实现的完整代码，仅为学习目的展示:
# 实际使用时只需上面两行 import 即可

# [3. EarlyStopping 类的实现]
class EarlyStopping:
    """
    Early Stopping 机制

    参数:
        patience:   连续 N 个 epoch 不降则停止训练
        min_delta:  多少变化算"改善"（防止微小波动触发）
        mode:       'min'（监控 loss 变小）或 'max'（监控 acc 变大）
        verbose:    是否打印停止信息
        path:       最优模型保存路径

    使用方式:
        early_stop = EarlyStopping(patience=5)
        for epoch in range(epochs):
            val_loss = train_and_eval()
            early_stop(val_loss, model, optimizer, epoch)
            if early_stop.early_stop:
                break
    """
    def __init__(self, patience=5, min_delta=0.0, mode='min',
                 verbose=True, path='../models/checkpoint/best_checkpoint.pth'):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.verbose = verbose
        self.path = path

        self.counter = 0            # 计数器：连续多少轮没有改善
        self.best_score = None      # 当前最佳得分
        self.early_stop = False     # 是否触发停止
        self.val_loss_min = np.inf  # 记录最小验证 loss

        # 确保保存目录存在
        os.makedirs(os.path.dirname(self.path), exist_ok=True)

    def __call__(self, score, model, optimizer, epoch):
        """
        每次 epoch 结束后调用

        参数:
            score:      当前 epoch 的验证指标（loss 或 acc）
            model:      模型（保存其 state_dict）
            optimizer:  优化器（保存其状态，用于断点续训）
            epoch:      当前 epoch 编号
        """
        # 根据 mode 转换 score（统一成"越大越好"）
        if self.mode == 'min':
            current_score = -score   # loss 越小 → -loss 越大
        else:
            current_score = score    # acc 越大越好

        # 第一轮：直接设为最优
        if self.best_score is None:
            self.best_score = current_score
            self._save_checkpoint(score, model, optimizer, epoch)
            return

        # 是否有显著改善？
        if current_score > self.best_score + self.min_delta:
            # 有改善 → 保存 checkpoint，重置计数器
            self.best_score = current_score
            self.counter = 0
            self._save_checkpoint(score, model, optimizer, epoch)
        else:
            # 无改善 → 计数器 +1
            self.counter += 1
            if self.verbose:
                print(f'  EarlyStopping: 计数器 {self.counter}/{self.patience} '
                      f'(当前最优验证 loss: {self.val_loss_min:.6f})')

            # 达到 patience → 停止训练
            if self.counter >= self.patience:
                self.early_stop = True
                if self.verbose:
                    print(f'\n  >>> 在第 {epoch} 个 epoch 触发 Early Stopping！')
                    print(f'  >>> 最优验证 loss: {self.val_loss_min:.6f}')

    def _save_checkpoint(self, score, model, optimizer, epoch):
        """保存当前最优 checkpoint（含模型 + 优化器状态）"""
        if self.mode == 'min':
            self.val_loss_min = score

        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_loss': self.val_loss_min if self.mode == 'min' else score,
        }, self.path)

        if self.verbose:
            print(f'  [checkpoint] 保存最优模型于 epoch {epoch} '
                  f'(验证 loss: {score:.6f})')


print('EarlyStopping 类定义完成！')

## 4. 训练/评估函数

与标准训练循环完全一致，额外返回每轮的训练/验证指标用于画图。

In [5]:
# [4. 训练/评估函数]
def train_one_epoch(model, loader, criterion, optimizer, device):
    """训练一个 epoch，返回 (avg_loss, accuracy)"""
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    """评估模型，返回 (avg_loss, accuracy)"""
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total


print('训练/评估函数定义完成！')

## 5. 对照实验：Early Stopping vs 跑满 100 epoch

我们用同一个 MLP 结构训练两次：

| 实验 | 策略 | 预期 |
|------|------|------|
| 实验 A | **Early Stopping**（patience=5） | ~20 epoch 自动停，验证准确率 ~98.5% |
| 实验 B | **跑满 100 epoch**（无早停） | 跑满 100 epoch，后期可能过拟合 |

> 你将看到：实验 A 用更少的训练时间，达到与实验 B 相当甚至更好的验证准确率。

In [6]:
# [5.1 实验 A：启用 Early Stopping]
print('=' * 50)
print('实验 A：Early Stopping (patience=5)')
print('=' * 50)

# ======== 超参数集中修改区 ========
LR = 0.001
WEIGHT_DECAY = 1e-4
EPOCHS = 100          # 最大 epoch 数，但 Early Stopping 会提前终止
PATIENCE = 5          # 连续 5 轮验证 loss 不降则停
CHECKPOINT_PATH = '../models/checkpoint/best_early_stop.pth'
# =================================

# 初始化新模型
model_a = MLP(hidden_dims=[512, 256]).to(device)
criterion = nn.CrossEntropyLoss()
optimizer_a = optim.Adam(model_a.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# 创建 Early Stopping 实例
early_stop = EarlyStopping(
    patience=PATIENCE, mode='min', min_delta=1e-4,
    verbose=True, path=CHECKPOINT_PATH
)

history_a = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(
        model_a, train_loader, criterion, optimizer_a, device
    )
    test_loss, test_acc = evaluate(
        model_a, test_loader, criterion, device
    )

    history_a['train_loss'].append(train_loss)
    history_a['train_acc'].append(train_acc)
    history_a['test_loss'].append(test_loss)
    history_a['test_acc'].append(test_acc)

    if epoch % 5 == 0 or epoch == 1:
        print(
            f'Epoch [{epoch:3d}/{EPOCHS}] '
            f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | '
            f'Test Loss:  {test_loss:.4f} | Test Acc:  {test_acc:.4f}'
        )

    # ======== 关键：每个 epoch 调用 Early Stopping ========
    early_stop(test_loss, model_a, optimizer_a, epoch)
    if early_stop.early_stop:
        break
    # ======================================================

print(f'\n实验 A 完成！共训练 {len(history_a["train_loss"])} 个 epoch')
print(f'最优验证 loss: {early_stop.val_loss_min:.6f}')

In [7]:
# [5.2 实验 B：跑满 100 epoch（无 Early Stopping）]
print('=' * 50)
print('实验 B：无 Early Stopping（跑满 100 轮）')
print('=' * 50)

# ======== 超参数集中修改区 ========
EPOCHS_B = 100
# =================================

# 初始化新模型（与实验 A 相同的初始化和结构）
model_b = MLP(hidden_dims=[512, 256]).to(device)
optimizer_b = optim.Adam(model_b.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

history_b = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}
best_b_acc = 0.0
best_b_epoch = 0

for epoch in range(1, EPOCHS_B + 1):
    train_loss, train_acc = train_one_epoch(
        model_b, train_loader, criterion, optimizer_b, device
    )
    test_loss, test_acc = evaluate(
        model_b, test_loader, criterion, device
    )

    history_b['train_loss'].append(train_loss)
    history_b['train_acc'].append(train_acc)
    history_b['test_loss'].append(test_loss)
    history_b['test_acc'].append(test_acc)

    if test_acc > best_b_acc:
        best_b_acc = test_acc
        best_b_epoch = epoch

    if epoch % 10 == 0 or epoch == 1:
        print(
            f'Epoch [{epoch:3d}/{EPOCHS_B}] '
            f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | '
            f'Test Loss:  {test_loss:.4f} | Test Acc:  {test_acc:.4f}'
        )

print(f'\n实验 B 完成！跑满 {EPOCHS_B} 个 epoch')
print(f'最佳测试准确率: {best_b_acc:.4f} (Epoch {best_b_epoch})')
print(f'最终测试准确率: {history_b["test_acc"][-1]:.4f} (Epoch {EPOCHS_B})')

## 6. 对照结果可视化

两张图展示 Early Stopping 的价值：
- **左图（实验 A）**：红色竖线标记 Early Stopping 触发点
- **右图（实验 B）**：绿色标记最佳 epoch，可以看到验证 loss 后期回升

In [8]:
# [6. 对照结果可视化]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ---------- 实验 A：Early Stopping ----------
epochs_a = range(1, len(history_a['train_loss']) + 1)
stop_epoch = len(epochs_a)

# A-左上：Loss 曲线
ax1 = axes[0, 0]
ax1.plot(epochs_a, history_a['train_loss'], 'b-', label='训练 Loss', alpha=0.7)
ax1.plot(epochs_a, history_a['test_loss'], 'r-', label='验证 Loss', linewidth=2)
ax1.axvline(x=stop_epoch, color='red', linestyle='--', alpha=0.6,
            label=f'Early Stop (Epoch {stop_epoch})')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title(f'实验 A：Early Stopping (patience={PATIENCE})\n停止于 epoch {stop_epoch}',
              fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# A-右上：Accuracy 曲线
ax2 = axes[0, 1]
ax2.plot(epochs_a, history_a['train_acc'], 'b-', label='训练 Acc', alpha=0.7)
ax2.plot(epochs_a, history_a['test_acc'], 'g-', label='验证 Acc', linewidth=2)
ax2.axvline(x=stop_epoch, color='red', linestyle='--', alpha=0.6,
            label=f'Early Stop')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title(f'实验 A：Accuracy | 最终验证 Acc = {history_a["test_acc"][-1]:.4f}',
              fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# ---------- 实验 B：无 Early Stopping ----------
epochs_b = range(1, len(history_b['train_loss']) + 1)

# B-左下：Loss 曲线
ax3 = axes[1, 0]
ax3.plot(epochs_b, history_b['train_loss'], 'b-', label='训练 Loss', alpha=0.7)
ax3.plot(epochs_b, history_b['test_loss'], 'r-', label='验证 Loss', linewidth=2)
ax3.axvline(x=best_b_epoch, color='green', linestyle='--', alpha=0.6,
            label=f'最佳 epoch {best_b_epoch}')
ax3.set_xlabel('Epoch')
ax3.set_ylabel('Loss')
ax3.set_title('实验 B：无 Early Stopping（跑满 100 epoch）\n'
             f'最终验证 Loss: {history_b["test_loss"][-1]:.4f} | '
             f'最佳验证 Loss: {min(history_b["test_loss"]):.4f}',
             fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)

# B-右下：Accuracy 曲线
ax4 = axes[1, 1]
ax4.plot(epochs_b, history_b['train_acc'], 'b-', label='训练 Acc', alpha=0.7)
ax4.plot(epochs_b, history_b['test_acc'], 'g-', label='验证 Acc', linewidth=2)
ax4.axvline(x=best_b_epoch, color='green', linestyle='--', alpha=0.6,
            label=f'最佳 epoch {best_b_epoch}')
ax4.set_xlabel('Epoch')
ax4.set_ylabel('Accuracy')
ax4.set_title(f'实验 B：Accuracy | 最佳 Acc = {best_b_acc:.4f} (epoch {best_b_epoch})\n'
             f'最终 Acc = {history_b["test_acc"][-1]:.4f} (epoch {EPOCHS_B})',
             fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 汇总对比
print('\n' + '=' * 60)
print('                     对照实验汇总')
print('=' * 60)
print(f'指标                    实验A(Early Stop)    实验B(跑满100)')
sep = '-' * 60
print(sep)
print(f'训练 epoch 数            {len(epochs_a):>10}             {len(epochs_b):>10}')
print(f'最佳验证 loss            {min(history_a["test_loss"]):>10.6f}         {min(history_b["test_loss"]):>10.6f}')
print(f'最佳验证 Acc             {max(history_a["test_acc"]):>10.4f}         {best_b_acc:>10.4f}')
print(f'最终验证 Acc             {history_a["test_acc"][-1]:>10.4f}         {history_b["test_acc"][-1]:>10.4f}')
print(f'训练时间（相对）         {len(epochs_a)/len(epochs_b)*100:>8.0f}%            ~100%')
print('=' * 60)

## 7. 加载 Checkpoint：从保存的最优模型恢复

Checkpoint 的核心价值：
1. **只保存最优模型**——不是最后一个 epoch，而是验证 loss 最低的那个
2. **含优化器状态**——可以从断点继续训练，而不仅是推理
3. **训练完成后加载最优权重做最终验证**

In [9]:
# [7. 加载 Checkpoint 并验证]
print('【加载 Checkpoint】')

# 1. 创建与保存时结构完全相同的模型
model_loaded = MLP(hidden_dims=[512, 256]).to(device)

# 2. 加载 checkpoint
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model_loaded.load_state_dict(checkpoint['model_state_dict'])

print(f'  从 epoch {checkpoint["epoch"]} 的 checkpoint 恢复')
print(f'  保存时的验证 loss: {checkpoint["best_val_loss"]:.6f}')

# 3. 在测试集上验证
model_loaded.eval()
criterion_eval = nn.CrossEntropyLoss()
test_loss, test_acc = evaluate(model_loaded, test_loader, criterion_eval, device)
print(f'\n  Checkpoint 模型测试结果:')
print(f'    Test Loss:  {test_loss:.6f}')
print(f'    Test Acc:   {test_acc:.4f} ({test_acc*100:.2f}%)')

# 4. 演示：从 checkpoint 继续训练（断点续训）
print(f'\n【演示：断点续训】')
print(f'  假设训练在第 {checkpoint["epoch"]} 个 epoch 中断，现在恢复...')

# 重建优化器并恢复其状态
optimizer_resume = optim.Adam(model_loaded.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
optimizer_resume.load_state_dict(checkpoint['optimizer_state_dict'])

# 继续训练 2 个 epoch
for epoch_extra in range(1, 3):
    train_loss, train_acc = train_one_epoch(
        model_loaded, train_loader, criterion_eval, optimizer_resume, device
    )
    test_loss2, test_acc2 = evaluate(
        model_loaded, test_loader, criterion_eval, device
    )
    print(f'  续训 Epoch {epoch_extra}: Train Acc={train_acc:.4f}, Test Acc={test_acc2:.4f}')

print(f'\n  断点续训完成！')
print(f'  这意味着：训练中断后，可以从上次保存的 checkpoint 完美恢复，不会丢失进度。')

> **提示：`EarlyStopping` 类已提取到 `utils/early_stopping.py`，**
> **你可以 `from utils.early_stopping import EarlyStopping` 直接在任何 notebook 中使用。**

## 8. 总结：Early Stopping 和 Checkpoint 的最佳实践

### 你应该记住的 5 个要点

| # | 要点 | 说明 |
|---|------|------|
| 1 | **早停监控验证 loss，不是训练 loss** | 训练 loss 永远在降，没有监控意义 |
| 2 | **patience=5~10 是经验值** | 太小容易误停（loss 震荡），太大浪费算力 |
| 3 | **保存 checkpoint 而不是保存最后模型** | 最后一个 epoch 的模型可能已经过拟合 |
| 4 | **checkpoint 要同时保存 optimizer state** | 这样断点续训时优化器动量/方差信息不会丢失 |
| 5 | **Early Stopping 节省的不仅是时间** | 它还是一种正则化——防止模型"死记硬背"训练集 |

### 训练循环的标准模板（含 Early Stopping）

```python
early_stop = EarlyStopping(patience=5, mode='min')

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(...)
    val_loss, val_acc = evaluate(...)

    # 一行代码：自动判断是否停止 + 保存最优模型
    early_stop(val_loss, model, optimizer, epoch)
    if early_stop.early_stop:
        break

# 训练结束后，加载最优 checkpoint 做最终评估
checkpoint = torch.load('best_checkpoint.pth')
model.load_state_dict(checkpoint['model_state_dict'])
```

### 和现有训练 notebook 的关系

你可以把 `EarlyStopping` 类**导入**到任何一个训练 notebook 中：
- `02_mlp_training.ipynb` — 替代手动的 `best_test_acc` 追踪（已内置）
- `05_cnn_mnist.ipynb` — CNN 也需要早停
- `06_resnet_mnist.ipynb` — ResNet 尤其需要，因为它训练久了更易过拟合
- 任何一个你自己的训练代码

> 这就是 Early Stopping + Checkpoint 的全部。简单、有效、每个深度学习项目都应该用。